# 31 — Trade Intelligence Analysis (Problem 3) — EDA

**Business question:** How dependent is each country on imports/exports?

**Decisions supported:**
- Diversify suppliers
- Find new export markets
- Reduce import dependency

**Scope:** 2001–2020 (per platform scope doc), `marts` schema.

**Tables used:**
- `fact_trade__matrix` (1986–2024) — bilateral (reporter/partner), import & export qty/value
- `fact_trade__trade` (1961–2023) — country-level, no partner breakdown
- `fact_trade__fertilizers_detailedtradematrix` (1990–2023) — bilateral, fertilizer-specific
- `fact_production__crops_livestock` (1961–2024) — needed to build a dependency ratio
  (e.g. imports ÷ (production + imports − exports)) since dependency isn't a raw field
  anywhere in the trade tables

**What this notebook does:**
1. Setup + schema confirmation
2. Row counts & year coverage (2001–2020)
3. Missingness per key column
4. Country/area coverage & join-key sanity checks across all 4 tables
5. What's available: element inventory for both the bilateral tables and the
   country-level table — need to see actual `element` values (Import/Export
   quantity vs. value) before assuming a dependency ratio is directly computable
6. A first pass at an import-dependency ratio for one country/item (proof of concept),
   plus a first pass at bilateral partner concentration (proof of concept)
7. Findings / gaps — feeds the locked KPI notebook (`32_trade_intelligence_analysis_kpi.ipynb`)

This is exploratory — same structure and lessons carried over from Problems 1 and 2:
confirm exact item/element strings from actual query output before hardcoding them,
filter out regional/income-group aggregates before any country-level ranking, and check
for the same `year` NULL / aggregate-contamination issues seen in `fact_food_security__data`
(may or may not affect these tables — checked in Sections 3 and 5).

**Note on scope decision:** both the bilateral tables (`fact_trade__matrix`,
`fact_trade__fertilizers_detailedtradematrix`) and the country-level table
(`fact_trade__trade`) are explored here side by side. The dependency-ratio KPI likely
uses the country-level or matrix-aggregated-to-country-level view; a partner-concentration
KPI (e.g. top-partner share of imports/exports) would need the bilateral detail. Final
KPI selection is deferred to `32_`, not decided here.

## 1. Setup

In [1]:
from _bootstrap import project_root
import polars as pl
import matplotlib.pyplot as plt

pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(200)
pl.Config.set_fmt_str_lengths(120)
pl.Config.set_tbl_rows(50)

from src.database.connection import get_duckdb_conn

conn = get_duckdb_conn(read_only=True)
print("Connected")

Connected


In [2]:
TABLES = [
    "fact_trade__matrix",
    "fact_trade__trade",
    "fact_trade__fertilizers_detailedtradematrix",
    "fact_production__crops_livestock",
]

YEAR_START, YEAR_END = 2001, 2020
SCHEMA = "marts"

## 2. Schema confirmation

In [3]:
existing = conn.execute(f"""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = '{SCHEMA}'
    ORDER BY table_name
""").fetchall()
existing_names = {t[0] for t in existing}

print(f"Tables found in '{SCHEMA}': {len(existing_names)}\n")
for t in TABLES:
    status = "OK" if t in existing_names else "MISSING"
    print(f"  [{status}] {t}")

Tables found in 'marts': 76

  [OK] fact_trade__matrix
  [OK] fact_trade__trade
  [OK] fact_trade__fertilizers_detailedtradematrix
  [OK] fact_production__crops_livestock


In [4]:
schemas = {}
for t in TABLES:
    schemas[t] = conn.execute(f'DESCRIBE SELECT * FROM {SCHEMA}."{t}"').pl()

for t in TABLES:
    print(f"--- {SCHEMA}.{t} ---")
    display(schemas[t])

--- marts.fact_trade__matrix ---


column_name,column_type,null,key,default,extra
str,str,str,str,str,str
"""reporter_country_code""","""BIGINT""","""YES""",null,null,null
"""reporter_country_code_m49""","""VARCHAR""","""YES""",null,null,null
"""reporter_countries""","""VARCHAR""","""YES""",null,null,null
"""partner_country_code""","""BIGINT""","""YES""",null,null,null
"""partner_country_code_m49""","""VARCHAR""","""YES""",null,null,null
"""partner_countries""","""VARCHAR""","""YES""",null,null,null
"""item_code""","""BIGINT""","""YES""",null,null,null
"""item_code_cpc""","""VARCHAR""","""YES""",null,null,null
"""item""","""VARCHAR""","""YES""",null,null,null


--- marts.fact_trade__trade ---


column_name,column_type,null,key,default,extra
str,str,str,str,str,str
"""area_code""","""VARCHAR""","""YES""",null,null,null
"""area_code_m49""","""VARCHAR""","""YES""",null,null,null
"""area""","""VARCHAR""","""YES""",null,null,null
"""item_code""","""VARCHAR""","""YES""",null,null,null
"""item""","""VARCHAR""","""YES""",null,null,null
"""element_code""","""VARCHAR""","""YES""",null,null,null
"""element""","""VARCHAR""","""YES""",null,null,null
"""year_code""","""BIGINT""","""YES""",null,null,null
"""year""","""BIGINT""","""YES""",null,null,null


--- marts.fact_trade__fertilizers_detailedtradematrix ---


column_name,column_type,null,key,default,extra
str,str,str,str,str,str
"""reporter_country_code""","""VARCHAR""","""YES""",null,null,null
"""reporter_country_code_m49""","""VARCHAR""","""YES""",null,null,null
"""reporter_countries""","""VARCHAR""","""YES""",null,null,null
"""partner_country_code""","""VARCHAR""","""YES""",null,null,null
"""partner_country_code_m49""","""VARCHAR""","""YES""",null,null,null
"""partner_countries""","""VARCHAR""","""YES""",null,null,null
"""item_code""","""VARCHAR""","""YES""",null,null,null
"""item_code_cpc""","""VARCHAR""","""YES""",null,null,null
"""item""","""VARCHAR""","""YES""",null,null,null


--- marts.fact_production__crops_livestock ---


column_name,column_type,null,key,default,extra
str,str,str,str,str,str
"""area_code""","""VARCHAR""","""YES""",null,null,null
"""area_code_m49""","""VARCHAR""","""YES""",null,null,null
"""area""","""VARCHAR""","""YES""",null,null,null
"""item_code""","""VARCHAR""","""YES""",null,null,null
"""item_code_cpc""","""VARCHAR""","""YES""",null,null,null
"""item""","""VARCHAR""","""YES""",null,null,null
"""element_code""","""VARCHAR""","""YES""",null,null,null
"""element""","""VARCHAR""","""YES""",null,null,null
"""year_code""","""BIGINT""","""YES""",null,null,null


### Notes — Schema confirmation

- **Confirmed: all 4 tables exist in `marts`** with the expected columns; nothing missing
  (76 tables found in `marts` total, all 4 targets OK).
- **Confirmed join-key type mismatch across the bilateral tables, and it matters:**
  `fact_trade__matrix` uses `reporter_country_code` / `partner_country_code` as
  **BIGINT**, while `fact_trade__fertilizers_detailedtradematrix` uses the identically
  named columns as **VARCHAR**. These two cannot be joined to each other on that key
  without an explicit CAST, and `fact_trade__matrix`'s BIGINT side also needs casting to
  join against `fact_socioeconomic__population.area_code` (VARCHAR) — confirmed
  necessary and applied in Section 5 below.
- `fact_trade__trade` uses a single `area_code` (VARCHAR), matching the Problem 1/2
  pattern — the "simple" table of the three trade tables, no cast needed to join it
  against `fact_production__crops_livestock` or `fact_socioeconomic__population`.

## 3. Row counts & year coverage (2001–2020)

In [5]:
coverage_rows = []
for t in TABLES:
    total = conn.execute(f'SELECT COUNT(*) FROM {SCHEMA}."{t}"').fetchone()[0]
    in_range = conn.execute(f"""
        SELECT COUNT(*) FROM {SCHEMA}."{t}"
        WHERE year BETWEEN {YEAR_START} AND {YEAR_END}
    """).fetchone()[0]
    null_year = conn.execute(f'SELECT COUNT(*) FROM {SCHEMA}."{t}" WHERE year IS NULL').fetchone()[0]
    year_min, year_max = conn.execute(f'SELECT MIN(year), MAX(year) FROM {SCHEMA}."{t}"').fetchone()
    coverage_rows.append({
        "table": t,
        "total_rows": total,
        "rows_2001_2020": in_range,
        "pct_in_scope": round(100 * in_range / total, 1) if total else None,
        "null_year_pct": round(100 * null_year / total, 2) if total else None,
        "year_min": year_min,
        "year_max": year_max,
    })

coverage_df = pl.DataFrame(coverage_rows)
coverage_df

table,total_rows,rows_2001_2020,pct_in_scope,null_year_pct,year_min,year_max
str,i64,i64,f64,f64,i64,i64
"""fact_trade__matrix""",52410630,33015677,63.0,0.0,1986,2024
"""fact_trade__trade""",190740,121745,63.8,0.0,1961,2023
"""fact_trade__fertilizers_detailedtradematrix""",5300910,3863454,72.9,0.0,1990,2023
"""fact_production__crops_livestock""",4209110,1433327,34.1,0.0,1961,2024


### Notes — Row counts & year coverage

- **No `year` NULL issue anywhere** — `null_year_pct` is 0.0% across all 4 tables. Same
  clean pattern as Problem 2; this is not a repeat of the Food Security year-parsing bug.
- Coverage in 2001–2020: `fact_trade__matrix` 63.0% (33,015,677 / 52,410,630 rows),
  `fact_trade__trade` 63.8% (121,745 / 190,740), `fact_trade__fertilizers_detailedtradematrix`
  72.9% (3,863,454 / 5,300,910), `fact_production__crops_livestock` 34.1% (already known
  from `21_`, re-confirmed unchanged here).
- **`fact_trade__matrix` is confirmed the largest table in the project by a wide margin**
  (52.4M total rows, 33M in scope) — the earlier concern about needing an aggregation
  step before this is usable in a notebook holds; direct SELECT/GROUP BY with a WHERE
  filter on `item`/`element`/`year` (as used in Section 7 below) keeps result sets small
  enough, but an unfiltered pull of the full table should be avoided.

## 4. Missingness per key column

In [6]:
def null_profile(table: str, columns: list[str]) -> pl.DataFrame:
    rows = []
    total = conn.execute(f'SELECT COUNT(*) FROM {SCHEMA}."{table}"').fetchone()[0]
    for col in columns:
        null_count = conn.execute(
            f'SELECT COUNT(*) FROM {SCHEMA}."{table}" WHERE "{col}" IS NULL'
        ).fetchone()[0]
        rows.append({
            "table": table,
            "column": col,
            "null_count": null_count,
            "null_pct": round(100 * null_count / total, 2) if total else None,
        })
    return pl.DataFrame(rows)

key_columns = {
    "fact_trade__matrix": [
        "reporter_country_code", "partner_country_code", "year", "value", "item", "element",
    ],
    "fact_trade__trade": ["area_code", "year", "value", "item", "element"],
    "fact_trade__fertilizers_detailedtradematrix": [
        "reporter_country_code", "partner_country_code", "year", "value", "item", "element",
    ],
    "fact_production__crops_livestock": ["area_code", "year", "value", "item", "element"],
}

null_dfs = [null_profile(t, cols) for t, cols in key_columns.items()]
pl.concat(null_dfs)

table,column,null_count,null_pct
str,str,i64,f64
"""fact_trade__matrix""","""reporter_country_code""",0,0.0
"""fact_trade__matrix""","""partner_country_code""",0,0.0
"""fact_trade__matrix""","""year""",0,0.0
"""fact_trade__matrix""","""value""",0,0.0
"""fact_trade__matrix""","""item""",0,0.0
"""fact_trade__matrix""","""element""",0,0.0
"""fact_trade__trade""","""area_code""",0,0.0
"""fact_trade__trade""","""year""",0,0.0
"""fact_trade__trade""","""value""",0,0.0


### Notes — Missingness

- **Zero missingness** in every key join/filter column (`area_code`,
  `reporter_country_code`, `partner_country_code`, `year`, `item`, `element`) across all
  4 tables — as clean as Problem 2.
- `value` is null in 2.24% of `fact_production__crops_livestock` rows (94,355 rows) —
  identical figure to what `21_` found, confirming no change in this table since then.
  All three trade tables show 0% null `value`.

## 5. Country/area coverage & join-key sanity checks

Same check as the Problem 1/2 EDAs — confirm join keys line up, and check whether these
trade tables also carry regional/income-group aggregate rows (as `fact_food_security__data`
did) before any country-level ranking. Extra check needed here versus prior EDAs: the two
bilateral tables use `reporter_country_code` / `partner_country_code` instead of a single
`area_code`, so the aggregate-exclusion join needs to run against **both** sides of each
bilateral row, not just one.

In [7]:
# fact_trade__trade uses the same area_code shape as Problems 1/2 — check directly.
area_col_map = {
    "fact_trade__trade": "area_code",
    "fact_production__crops_livestock": "area_code",
}

rows = []
for t, col in area_col_map.items():
    n = conn.execute(f'SELECT COUNT(DISTINCT "{col}") FROM {SCHEMA}."{t}"').fetchone()[0]
    rows.append({"table": t, "area_column": col, "distinct_areas": n})

pl.DataFrame(rows)

table,area_column,distinct_areas
str,str,i64
"""fact_trade__trade""","""area_code""",294
"""fact_production__crops_livestock""","""area_code""",244


In [8]:
# Bilateral tables: check distinct reporter and partner counts separately —
# a mismatch between the two would suggest asymmetric coverage (e.g. a country only
# ever appears as a partner, never as a reporter).
bilateral_tables = ["fact_trade__matrix", "fact_trade__fertilizers_detailedtradematrix"]

rows = []
for t in bilateral_tables:
    n_reporters = conn.execute(f'SELECT COUNT(DISTINCT reporter_country_code) FROM {SCHEMA}."{t}"').fetchone()[0]
    n_partners = conn.execute(f'SELECT COUNT(DISTINCT partner_country_code) FROM {SCHEMA}."{t}"').fetchone()[0]
    rows.append({"table": t, "distinct_reporters": n_reporters, "distinct_partners": n_partners})

pl.DataFrame(rows)

table,distinct_reporters,distinct_partners
str,i64,i64
"""fact_trade__matrix""",195,220
"""fact_trade__fertilizers_detailedtradematrix""",217,247


In [9]:
# Aggregate-contamination check, mirroring Problems 1/2 — but here run against
# fact_trade__trade's area_code AND against fact_trade__matrix's reporter_country_code,
# since the bilateral table's join-key type (BIGINT) differs from population's (VARCHAR)
# and needs an explicit cast to compare.
missing_from_population_trade = conn.execute(f"""
    SELECT DISTINCT tr.area_code, tr.area
    FROM {SCHEMA}.fact_trade__trade tr
    LEFT JOIN {SCHEMA}.fact_socioeconomic__population pop
        ON tr.area_code = pop.area_code
    WHERE pop.area_code IS NULL
    ORDER BY tr.area
""").pl()

print(f"fact_trade__trade areas with NO match in population data: {missing_from_population_trade.height}")
missing_from_population_trade

fact_trade__trade areas with NO match in population data: 29


area_code,area
str,str
"""51000""","""Africa (excluding intra-trade)"""
"""52000""","""Americas (excluding intra-trade)"""
"""53000""","""Asia (excluding intra-trade)"""
"""55010""","""Australia and New Zealand (excluding intra-trade)"""
"""52060""","""Caribbean (excluding intra-trade)"""
"""52040""","""Central America (excluding intra-trade)"""
"""53010""","""Central Asia (excluding intra-trade)"""
"""51010""","""Eastern Africa (excluding intra-trade)"""
"""53020""","""Eastern Asia (excluding intra-trade)"""


In [10]:
missing_from_population_matrix = conn.execute(f"""
    SELECT DISTINCT m.reporter_country_code, m.reporter_country_name
    FROM {SCHEMA}.fact_trade__matrix m
    LEFT JOIN {SCHEMA}.fact_socioeconomic__population pop
        ON CAST(m.reporter_country_code AS VARCHAR) = pop.area_code
    WHERE pop.area_code IS NULL
    ORDER BY m.reporter_country_name
""").pl()

print(f"fact_trade__matrix reporter areas with NO match in population data: {missing_from_population_matrix.height}")
missing_from_population_matrix

fact_trade__matrix reporter areas with NO match in population data: 0


reporter_country_code,reporter_country_name
i64,str


In [11]:
# Partner-side check — the reporter side of fact_trade__matrix was confirmed clean
# (0 unmatched) above, but the partner side was never actually checked. If a partner
# code is a regional/income-group aggregate (like the 29 found in fact_trade__trade),
# it would silently inflate any bilateral concentration/diversification KPI (Section 7,
# and 32_ KPI 2/3) by letting a non-country "partner" win a top-partner or
# significant-partner slot.
missing_from_population_matrix_partners = conn.execute(f"""
    SELECT DISTINCT m.partner_country_code, m.partner_country_name
    FROM {SCHEMA}.fact_trade__matrix m
    LEFT JOIN {SCHEMA}.fact_socioeconomic__population pop
        ON CAST(m.partner_country_code AS VARCHAR) = pop.area_code
    WHERE pop.area_code IS NULL
    ORDER BY m.partner_country_name
""").pl()

print(f"fact_trade__matrix partner areas with NO match in population data: {missing_from_population_matrix_partners.height}")
missing_from_population_matrix_partners

fact_trade__matrix partner areas with NO match in population data: 9


partner_country_code,partner_country_name
i64,str
31,"""Bouvet Island"""
34,"""Canton and Enderbury Islands"""
92,"""Heard Island and McDonald Islands"""
111,"""Johnston Island"""
139,"""Midway Island"""
271,"""South Georgia and the South Sandwich Islands"""
260,"""Svalbard and Jan Mayen Islands"""
232,"""United States Minor Outlying Islands"""
242,"""Wake Island"""


### Notes — Country/area coverage & join keys

- Distinct area counts: `fact_trade__trade` 294 areas, `fact_production__crops_livestock`
  244 areas (matches `21_`). Bilateral tables: `fact_trade__matrix` 195 reporters / 220
  partners, `fact_trade__fertilizers_detailedtradematrix` 217 reporters / 247 partners —
  the partner side is consistently larger than the reporter side in both bilateral
  tables, which makes sense (small/non-reporting countries still show up as trade
  partners of larger reporting countries).
- **`fact_trade__trade` DOES carry regional/aggregate contamination — confirmed 29
  unmatched areas** against `fact_socioeconomic__population` (e.g. "Africa (excluding
  intra-trade)", "Eastern Europe (excluding intra-trade)", continent/sub-region
  aggregates, plus a couple of small non-independent territories like Norfolk Island and
  Pitcairn that simply aren't in the population table). **This means, unlike Problem 2,
  Problem 3's country-level dependency KPI needs the same aggregate-exclusion join used
  in Problem 1** — filter `fact_trade__trade` to only areas that also appear in
  `fact_socioeconomic__population` before any country-level ranking or Power BI display.
- **`fact_trade__matrix`, by contrast, shows 0 unmatched reporters** once the BIGINT→VARCHAR
  cast is applied — no aggregate-exclusion join needed on the bilateral table's reporter
  side. **Partner side now checked (new cell above): confirm the printed count before
  trusting any bilateral concentration/diversification KPI (32_ KPI 2/3) — if it comes
  back non-zero, those KPIs need the same partner-side aggregate-exclusion filter that
  KPI 1 already applies to `fact_trade__trade`.**

## 6. What's available: import/export elements across the trade tables

All three trade tables are long/normalized — need to see the actual `element` values to
know what's directly available (Import/Export quantity, Import/Export value) versus what
needs deriving (a dependency ratio isn't a raw field anywhere).

In [12]:
trade_elements = conn.execute(f"""
    SELECT element, COUNT(*) AS n_rows, COUNT(DISTINCT area_code) AS n_countries,
           COUNT(DISTINCT item) AS n_items, MIN(year) AS year_min, MAX(year) AS year_max
    FROM {SCHEMA}.fact_trade__trade
    WHERE year BETWEEN {YEAR_START} AND {YEAR_END}
    GROUP BY element
    ORDER BY n_rows DESC
""").pl()

print("fact_trade__trade — element breakdown:")
trade_elements

fact_trade__trade — element breakdown:


element,n_rows,n_countries,n_items,year_min,year_max
str,i64,i64,i64,i64,i64
"""Import value""",37012,280,37,2001,2020
"""Import quantity""",37001,280,37,2001,2020
"""Export value""",23867,275,37,2001,2020
"""Export quantity""",23865,275,37,2001,2020


In [13]:
# fact_trade__trade only has 37 distinct items total (confirmed above) — list them all
# to pick a real representative item, since "Wheat" is confirmed NOT among them.
trade_items = conn.execute(f"""
    SELECT item, COUNT(*) AS n_rows, COUNT(DISTINCT area_code) AS n_countries,
           MIN(year) AS year_min, MAX(year) AS year_max
    FROM {SCHEMA}.fact_trade__trade
    WHERE year BETWEEN {YEAR_START} AND {YEAR_END}
    GROUP BY item
    ORDER BY n_rows DESC
""").pl()

print(f"fact_trade__trade — all {trade_items.height} distinct items, most-covered first:")
trade_items

fact_trade__trade — all 37 distinct items, most-covered first:


item,n_rows,n_countries,year_min,year_max
str,i64,i64,i64,i64
"""Pesticides (total)""",21736,280,2001,2020
"""Insecticides (excl. Haz. pest.)""",8376,218,2001,2020
"""Disinfectants, etc (excl. Haz. pest.)""",8194,218,2007,2020
"""Herbicides (excl. Haz. pest.)""",7288,208,2007,2020
"""Fungicides (excl. Haz. pest.)""",7110,206,2007,2020
"""Hazardous pesticides""",6356,212,2007,2020
"""Insecticides""",4990,213,2001,2019
"""Oxirane (ethylene oxide)""",4718,159,2001,2020
"""Disinfectants, etc""",4698,212,2001,2019


In [14]:
# Cross-reference: for each of fact_trade__trade's 37 items, check whether the SAME
# exact item string also exists in fact_trade__matrix and in
# fact_production__crops_livestock. This directly answers the open question from
# Section 6/7 (item universes may not overlap by name) instead of discovering it by
# trial and error when 32_'s KPI 2/3 queries come back empty.
# Run this AFTER the cell above so trade_items is available.
trade_item_list = trade_items["item"].to_list()

matrix_item_check = conn.execute(f"""
    SELECT item, COUNT(*) AS n_rows, COUNT(DISTINCT reporter_country_code) AS n_reporters
    FROM {SCHEMA}.fact_trade__matrix
    WHERE item = ANY(?)
      AND year BETWEEN {YEAR_START} AND {YEAR_END}
    GROUP BY item
""", [trade_item_list]).pl()

production_item_check = conn.execute(f"""
    SELECT item, COUNT(*) AS n_rows, COUNT(DISTINCT area_code) AS n_countries
    FROM {SCHEMA}.fact_production__crops_livestock
    WHERE item = ANY(?)
      AND element = 'Production'
      AND year BETWEEN {YEAR_START} AND {YEAR_END}
    GROUP BY item
""", [trade_item_list]).pl()

item_overlap = (
    trade_items
    .select(["item", "n_rows", "n_countries"])
    .rename({"n_rows": "trade_n_rows", "n_countries": "trade_n_countries"})
    .join(
        matrix_item_check.rename({"n_rows": "matrix_n_rows", "n_reporters": "matrix_n_reporters"}),
        on="item", how="left"
    )
    .join(
        production_item_check.rename({"n_rows": "production_n_rows", "n_countries": "production_n_countries"}),
        on="item", how="left"
    )
    .sort("trade_n_rows", descending=True)
)

print(f"Items also found in fact_trade__matrix: {item_overlap.filter(pl.col('matrix_n_rows').is_not_null()).height} / {item_overlap.height}")
print(f"Items also found in fact_production__crops_livestock: {item_overlap.filter(pl.col('production_n_rows').is_not_null()).height} / {item_overlap.height}")
item_overlap

Items also found in fact_trade__matrix: 0 / 37
Items also found in fact_production__crops_livestock: 0 / 37


item,trade_n_rows,trade_n_countries,matrix_n_rows,matrix_n_reporters,production_n_rows,production_n_countries
str,i64,i64,i64,i64,i64,i64
"""Pesticides (total)""",21736,280,null,null,null,null
"""Insecticides (excl. Haz. pest.)""",8376,218,null,null,null,null
"""Disinfectants, etc (excl. Haz. pest.)""",8194,218,null,null,null,null
"""Herbicides (excl. Haz. pest.)""",7288,208,null,null,null,null
"""Fungicides (excl. Haz. pest.)""",7110,206,null,null,null,null
"""Hazardous pesticides""",6356,212,null,null,null,null
"""Insecticides""",4990,213,null,null,null,null
"""Oxirane (ethylene oxide)""",4718,159,null,null,null,null
"""Disinfectants, etc""",4698,212,null,null,null,null


In [15]:
matrix_elements = conn.execute(f"""
    SELECT element, COUNT(*) AS n_rows,
           COUNT(DISTINCT reporter_country_code) AS n_reporters,
           COUNT(DISTINCT partner_country_code) AS n_partners,
           COUNT(DISTINCT item) AS n_items,
           MIN(year) AS year_min, MAX(year) AS year_max
    FROM {SCHEMA}.fact_trade__matrix
    WHERE year BETWEEN {YEAR_START} AND {YEAR_END}
    GROUP BY element
    ORDER BY n_rows DESC
""").pl()

print("fact_trade__matrix — element breakdown:")
matrix_elements

fact_trade__matrix — element breakdown:


element,n_rows,n_reporters,n_partners,n_items,year_min,year_max
str,i64,i64,i64,i64,i32,i32
"""Import value""",8414302,186,204,540,2001,2020
"""Import quantity""",8339036,186,204,539,2001,2020
"""Export value""",8162508,185,207,542,2001,2020
"""Export quantity""",8099831,185,207,541,2001,2020


In [16]:
fert_matrix_elements = conn.execute(f"""
    SELECT element, COUNT(*) AS n_rows,
           COUNT(DISTINCT reporter_country_code) AS n_reporters,
           COUNT(DISTINCT partner_country_code) AS n_partners,
           COUNT(DISTINCT item) AS n_items,
           MIN(year) AS year_min, MAX(year) AS year_max
    FROM {SCHEMA}.fact_trade__fertilizers_detailedtradematrix
    WHERE year BETWEEN {YEAR_START} AND {YEAR_END}
    GROUP BY element
    ORDER BY n_rows DESC
""").pl()

print("fact_trade__fertilizers_detailedtradematrix — element breakdown:")
fert_matrix_elements

fact_trade__fertilizers_detailedtradematrix — element breakdown:


element,n_rows,n_reporters,n_partners,n_items,year_min,year_max
str,i64,i64,i64,i64,i64,i64
"""Import quantity""",857446,213,233,25,2001,2020
"""Export quantity""",849560,207,240,25,2001,2020
"""Import value""",474192,213,232,22,2001,2020
"""Export value""",471426,207,236,22,2001,2020
"""Import quantity (tonnes N)""",298303,213,228,13,2001,2020
"""Export quantity (tonnes N)""",296759,205,235,13,2001,2020
"""Import quantity (tonnes P)""",156274,212,216,9,2001,2020
"""Export quantity (tonnes P)""",155380,198,232,9,2001,2020
"""Import quantity (tonnes K)""",152545,213,213,6,2001,2020


In [17]:
# Production elements were already profiled in 21_ — re-check here only to confirm
# "Production" is still present with the same coverage, since it's needed as the
# denominator for a dependency ratio (imports / (production + imports - exports)).
production_check = conn.execute(f"""
    SELECT element, COUNT(*) AS n_rows, COUNT(DISTINCT area_code) AS n_countries,
           MIN(year) AS year_min, MAX(year) AS year_max
    FROM {SCHEMA}.fact_production__crops_livestock
    WHERE year BETWEEN {YEAR_START} AND {YEAR_END}
      AND element = 'Production'
    GROUP BY element
""").pl()

production_check

element,n_rows,n_countries,year_min,year_max
str,i64,i64,i64,i64
"""Production""",553548,239,2001,2020


### Notes — Available elements

- **Both quantity and value are available for Import and Export, in all three trade
  tables** — confirmed via Cells 21–25:
  - `fact_trade__trade`: `Import value` (37,012 rows / 280 countries), `Import quantity`
    (37,001 / 280), `Export value` (23,867 / 275), `Export quantity` (23,865 / 275) —
    **only 37 distinct items total**, full 2001–2020 coverage.
  - `fact_trade__matrix`: `Import value` (8.4M rows / 186 reporters / 204 partners),
    `Import quantity` (8.3M / 186 / 204), `Export value` (8.2M / 185 / 207),
    `Export quantity` (8.1M / 185 / 207) — ~540 distinct items, full 2001–2020 coverage.
  - `fact_trade__fertilizers_detailedtradematrix`: same four base elements plus
    nutrient-specific splits (`Import/Export quantity (tonnes N/P/K)`) — only 25 distinct
    items (fertilizer products), full 2001–2020 coverage.
- **IMPORTANT — element string casing confirmed from real output.** The actual strings
  are **`"Import value"`**, `"Import quantity"`, `"Export value"`, `"Export quantity"` —
  only the first word is capitalized. This was originally guessed wrong (Title Case) and
  caused a `ColumnNotFoundError` downstream in the KPI notebook — now fixed everywhere.
- **CONFIRMED — `fact_trade__trade` is NOT crop-level detail; it's a small, curated
  37-item commodity list.** A dedicated item-listing query (Cell 22, run after this
  section originally shipped) confirmed zero items match "wheat" in any form — this
  isn't a naming mismatch (e.g. "Wheat" vs. "Wheat and meslin"), it's a genuine scope
  limitation: this table only covers 37 total commodities across all of 2001–2020,
  and wheat specifically isn't one of them. **KPI 1's representative item must be chosen
  from Cell 22's real 37-item list, not assumed to match Problems 1/2's Wheat choice.**
  `fact_trade__matrix`, by contrast, has ~540 items and likely does include wheat (or a
  wheat-adjacent CPC code) — but it's bilateral-only, so a country-level total there
  needs summing across all partners first.

## 7. First pass: import-dependency ratio + partner concentration (proof of concept)

**Revised after first run:** the original pass through this section used
`test_item = "Wheat"` and element strings `'Import Quantity'` / `'Export Quantity'`
(Title Case) as placeholders. Both were wrong — Section 6 confirmed the real element
strings are `"Import value"` / `"Import quantity"` / `"Export value"` /
`"Export quantity"` (lowercase second word), and a plain `"Wheat"` returned zero rows
from `fact_trade__trade`. Cell 27 below now starts with a real item-discovery query
instead of assuming the item string.

Two separate proofs of concept, since the dependency ratio and partner concentration
draw on different table shapes:
1. Import dependency ratio for one country/item, using `fact_trade__trade` (quantity)
   joined to `fact_production__crops_livestock` (Production).
2. Partner concentration (top-partner share of imports) for one country/item, using
   `fact_trade__matrix` bilateral detail — this is the piece `fact_trade__trade` alone
   can't answer.

In [18]:
# Item-discovery step (added after the first run showed "Wheat" returns 0 rows from
# fact_trade__trade). Confirmed via Cell 22: fact_trade__trade has only 37 distinct
# items total, and none match "wheat" in any form -- Cell 22 above shows the real list.
# Search fact_production__crops_livestock for wheat separately, for reference only --
# it is NOT expected to overlap with fact_trade__trade's 37-item list.
wheat_items_production = conn.execute(f"""
    SELECT DISTINCT item FROM {SCHEMA}.fact_production__crops_livestock
    WHERE LOWER(item) LIKE '%wheat%'
    ORDER BY item
""").pl()

print("fact_production__crops_livestock — items matching 'wheat' (for reference; "
      "fact_trade__trade has no equivalent, per Cell 22):")
wheat_items_production

fact_production__crops_livestock — items matching 'wheat' (for reference; fact_trade__trade has no equivalent, per Cell 22):


item
str
"""Buckwheat"""
"""Wheat"""


In [19]:
# CONFIRMED from a real run of the item-listing cell above: fact_trade__trade's 37
# items are all pesticides/agrochemicals, not crops. "Pesticides (total)" is the
# best-covered item (21,736 rows, 280 countries, full 2001-2020 range) and is used here.
# NOTE: this item has no matching row in fact_production__crops_livestock's Production
# element (that table is crop/livestock output, not pesticide manufacturing) -- Cell 30
# below (the production join) is expected to return an empty/near-empty result for this
# reason, which 32_'s KPI 1 accounts for by dropping the production join entirely.
test_item = "Pesticides (total)"

trade_test = conn.execute(f"""
    SELECT area_code, area, year, element, value
    FROM {SCHEMA}.fact_trade__trade
    WHERE item = ?
      AND element IN ('Import quantity', 'Export quantity')
      AND year BETWEEN {YEAR_START} AND {YEAR_END}
""", [test_item]).pl().with_columns(pl.col("value").cast(pl.Float64, strict=False))

print(f"Rows for item='{test_item}' in fact_trade__trade: {trade_test.height}")
trade_test.head(10)

Rows for item='Pesticides (total)' in fact_trade__trade: 10868


area_code,area,year,element,value
str,str,i64,str,f64
"""3""","""Albania""",2001,"""Import quantity""",1954.923
"""3""","""Albania""",2002,"""Import quantity""",1420.618
"""3""","""Albania""",2003,"""Import quantity""",1903.808
"""3""","""Albania""",2004,"""Import quantity""",1789.615
"""3""","""Albania""",2005,"""Import quantity""",1758.334
"""3""","""Albania""",2006,"""Import quantity""",1286.574
"""3""","""Albania""",2007,"""Import quantity""",1942.615
"""3""","""Albania""",2008,"""Import quantity""",1607.214
"""3""","""Albania""",2009,"""Import quantity""",1295.727


In [20]:
# NOTE: test_item = "Pesticides (total)" is a pesticide, not a crop -- this join
# against fact_production__crops_livestock's Production element is EXPECTED to return
# 0 (or near-0) rows, since that table has no "production" figure for a pesticide. This
# confirms, rather than contradicts, the finding above -- it's why 32_'s KPI 1 was
# redefined to a pure trade-based ratio (imports / (imports+exports)) that doesn't need
# this join at all. Kept here as a proof-of-concept record, not a bug to fix.
production_test = conn.execute(f"""
    SELECT area_code, area, year, value AS production_value
    FROM {SCHEMA}.fact_production__crops_livestock
    WHERE item = ?
      AND element = 'Production'
      AND year BETWEEN {YEAR_START} AND {YEAR_END}
""", [test_item]).pl()

trade_pivot = trade_test.pivot(values="value", index=["area_code", "area", "year"], on="element")

dependency_test = trade_pivot.join(production_test, on=["area_code", "area", "year"], how="inner")
print(f"Joined rows: {dependency_test.height}")
dependency_test.head(10)

Joined rows: 0


area_code,area,year,Import quantity,Export quantity,production_value
str,str,i64,f64,f64,f64


In [21]:
# Proof of concept 2: bilateral partner concentration — top partner's share of a
# country's imports for one item/year, using fact_trade__matrix.
# fact_trade__matrix has ~540 distinct items (vs. fact_trade__trade's 37) -- test_item
# chosen from fact_trade__trade's list above may or may not exist here under the same
# name; if this returns 0 rows, search fact_trade__matrix's items separately rather
# than assuming test_item carries over.
test_reporter_name = "Afghanistan"  # swap for a larger/more-traded country if this returns few rows

partner_test = conn.execute(f"""
    SELECT reporter_country_name, partner_country_name, year, value
    FROM {SCHEMA}.fact_trade__matrix
    WHERE item = ?
      AND element = 'Import quantity'
      AND reporter_country_name = ?
      AND year BETWEEN {YEAR_START} AND {YEAR_END}
    ORDER BY year, value DESC
""", [test_item, test_reporter_name]).pl()

print(f"Rows for reporter='{test_reporter_name}', item='{test_item}': {partner_test.height}")
partner_test.head(10)

Rows for reporter='Afghanistan', item='Pesticides (total)': 0


reporter_country_name,partner_country_name,year,value
str,str,i32,f64


### Notes — Proof of concept

- **Confirmed, real finding: `fact_trade__trade` does not contain wheat under any name.**
  Cell 22's full 37-item list (run for real) is the source of truth here — Cell 28 above
  only checks the production side for reference, since checking `fact_trade__trade` for
  "wheat" again would be redundant (already confirmed empty).
- **TODO after running Cell 22 for real and choosing an item from its output:**
  - Set `test_item` in Cell 29 to a real item from `fact_trade__trade`'s list — prefer
    one with high `n_rows`/`n_countries` for a clean proof of concept.
  - Record whether Cell 30's join (against `fact_production__crops_livestock`) produces
    a reasonable row count. `fact_trade__trade`'s 37 items may be broader trade
    categories (e.g. "Cereals, nes", "Oilseeds nes") rather than exact crop names — if
    so, this join may need a different key (e.g. matching by `item_code_cpc` instead of
    the free-text `item` string) rather than a direct name match.
  - Record whether Cell 31's partner-concentration query returns rows — `test_item` was
    chosen from `fact_trade__trade`'s smaller item universe, so it may not exist by the
    same name in `fact_trade__matrix`'s ~540-item list; if it returns 0 rows, search
    `fact_trade__matrix` separately for a matching item.

## 8. Findings & gaps — what's ready for the KPI notebook

### What's ready as-is
- All 4 tables: zero year-nulls, zero missingness in key columns. Same clean pattern as
  Problem 2 on those two dimensions.
- All three trade tables provide both quantity and value for Import and Export directly
  — no derivation needed to get raw trade flows themselves. Real element strings
  confirmed: `"Import value"`, `"Import quantity"`, `"Export value"`, `"Export quantity"`.

### What's NOT clean, unlike Problem 2 — carry these into the KPI notebook
- **`fact_trade__trade` has regional/aggregate contamination** (29 unmatched areas
  against `fact_socioeconomic__population`, confirmed in Section 5) — needs the same
  aggregate-exclusion join used in Problem 1.
- **Join-key type mismatch across bilateral tables**: `fact_trade__matrix`'s
  reporter/partner codes are BIGINT, `fact_trade__fertilizers_detailedtradematrix`'s are
  VARCHAR — any cross-join needs an explicit CAST (confirmed necessary, Section 5).
- **`fact_trade__trade` is a curated 37-item commodity list, not full crop-level
  detail — confirmed, not a naming mismatch.** Wheat is absent entirely. Any KPI built on
  this table must pick its representative item from the real 37, and should not assume
  parity with Problems 1/2's Wheat choice. This also raises a real possibility that
  `fact_trade__trade`'s items are broader commodity groupings than
  `fact_production__crops_livestock`'s crop-level items — the join between the two
  (needed for KPI 1's dependency ratio) may not be a clean 1:1 name match and could need
  `item_code_cpc` instead of the free-text `item` string. This needs confirming once a
  real item is chosen and the Section 7 join is actually run.

### What needs a derived metric (candidate for `32_trade_intelligence_analysis_kpi.ipynb`)
1. **Import dependency ratio** (imports ÷ apparent domestic supply) — query pattern
   confirmed workable in principle, pending: (a) a real item chosen from
   `fact_trade__trade`'s 37-item list, (b) confirming that item also exists (by the same
   or a matchable name) in `fact_production__crops_livestock`, (c) the aggregate-exclusion
   join already confirmed necessary.
2. **Partner concentration / diversification measure** — built on `fact_trade__matrix`'s
   ~540-item bilateral detail, which is a different (larger) item universe than
   `fact_trade__trade`'s 37 — the chosen item may need to differ between KPI 1 and
   KPI 2/3 rather than being shared.

### Data quality issues to flag
- Regional-aggregate contamination in `fact_trade__trade` (confirmed, 29 areas).
- No `year` NULL issue, no missingness issue.
- Bilateral join-key type mismatch (BIGINT vs. VARCHAR) — requires explicit CAST.
- `fact_trade__trade`'s narrow 37-item scope — not a data quality bug, but a real scope
  constraint that changes which crops/commodities Problem 3's KPIs can represent at the
  country-level-totals granularity.

### Open questions for the next notebook / dbt work
- **Immediate, blocking for `32_`:** run the item-listing cell (Section 6) and the new
  item cross-reference cell right after it for real. The cross-reference cell answers
  the "same item or different items?" question directly — pick `test_item` from
  `item_overlap`'s output, preferring a row with high `trade_n_rows` where
  `production_n_rows` is also populated (needed for KPI 1's join). If no single item
  satisfies both KPI 1 (trade+production) and KPI 2/3 (trade+matrix), note the best
  candidate for each separately in `32_`.
- **New, also blocking for `32_` KPI 2/3:** run the new partner-side aggregate check
  (Section 5, added after the reporter-side check) — if it returns a non-zero count,
  `32_`'s KPI 2 (top-partner share) and KPI 3 (significant-partner count) need the same
  aggregate-exclusion filter KPI 1 already applies, or a regional aggregate could win a
  "top partner" slot.
- Quantity vs. value basis for the dependency ratio — both confirmed available.
- Should `fact_trade__fertilizers_detailedtradematrix` feed its own dedicated KPI, or
  fold into a general dependency ratio as one more traded category?

## Close connection

In [22]:
conn.close()
print("Connection closed")

Connection closed
